# 📏 长上下文 — 为什么受限，又如何突破 1M

**前置阅读**：建议先读完 `01-theory/02-transformer-architecture.ipynb`（理解 Attention 机制）和 `01-theory/03-transformer-inference.ipynb`（理解 KV Cache）。

**本文目标**：系统理解上下文长度的三大瓶颈，以及当前主流模型突破 1M 上下文的核心技术。

读完这篇你会理解：
- Attention O(n²) 到底有多可怕（128K → 1M 的计算量飞跃）
- KV Cache 为什么是"沉默的显存杀手"
- RoPE 外推为什么失效，PI/NTK/YaRN 如何修复
- Ring Attention、稀疏注意力、KV Cache 量化的分工协作

## 1. 三大瓶颈全景

```
上下文长度从 4K → 128K → 1M，每步跨越都撞上不同的墙:

             4K → 32K             32K → 128K            128K → 1M
             ─────────            ──────────            ───────────
瓶颈 1:
Attention    O(n²) 可忽略         开始感知               严重瓶颈
计算量       0.016B ops/head      1B ops/head            64B ops/head

瓶颈 2:
KV Cache     ~2 MB/请求           ~64 MB/请求            ~500 MB/请求
显存         无压力                需要关注               ← 单请求就 500MB!

瓶颈 3:
位置编码     训练长度内完美         需要 RoPE 扩展         需要全新的位置方案
外推         无问题                (PI/NTK/YaRN)         (或稀疏注意力绕开)
```

### 1.1 瓶颈在不同阶段的表现差异

```
Prefill 阶段 (一次处理全部 prompt):
  瓶颈 1 (计算) → 主要矛盾 ← O(n²) 的 Attention 计算
  瓶颈 2 (显存) → 次要矛盾 (KV Cache 写入是一次性的)
  瓶颈 3 (位置) → 不涉及 (prefill 时位置编码在训练长度内)

Decode 阶段 (逐 token 生成):
  瓶颈 1 (计算) → 次要矛盾 ← Decode 是 O(n), 不是 O(n²)
  瓶颈 2 (显存) → 主要矛盾 ← 每步都要读全部 KV Cache
  瓶颈 3 (位置) → 可能涉及 (如果生成超过训练长度)
```

这解释了为什么不同优化技术针对不同阶段。

## 2. 瓶颈一: Attention 的 O(n²) 计算

### 2.1 具体数字

```python
# 单个 Attention Head 的 FLOPs (简化)
def attention_flops(seq_len, d_head=128):
    # Q @ K^T: [n, d] @ [d, n] = n² × d
    scores = seq_len * seq_len * d_head
    # softmax: ~5 × n² (指数 + 求和 + 除法)
    softmax_ops = 5 * seq_len * seq_len
    # attn @ V: [n, n] @ [n, d] = n² × d
    output = seq_len * seq_len * d_head
    return scores + softmax_ops + output
```

| Context | 每头 FLOPs | 32 头总 FLOPs | A100 理论耗时 |
|---------|-----------|-------------|-------------|
| 4K | 0.016B | 0.5B | ~0.001s |
| 32K | 1B | 32B | ~0.05s |
| 128K | 16B | 512B | ~0.8s |
| 256K | 64B | 2T | ~3.2s |
| 1M | 1T | 32T | **~50s** |

> A100 FP16 理论算力 312 TFLOPS，实际利用率 ~20% for Attention

### 2.2 Prefill 才是重灾区

为什么 Decode 阶段不受 O(n²) 影响？

```
Prefill (处理 prompt，一次性):
  Q: [n, d]   ← 所有 prompt token
  K: [n, d]   ← 所有 prompt token
  Q @ K^T: [n, d] @ [d, n] → [n, n]  ← O(n²)!

Decode (逐 token 生成):
  Q: [1, d]   ← 只有新 token 的 Q
  K: [n, d]   ← 历史全部 KV Cache
  Q @ K^T: [1, d] @ [d, n] → [1, n]  ← O(n), 不是 O(n²)!

所以长上下文推理的 Prefill 延迟是主要矛盾，
而 Decode 的瓶颈在显存带宽 (memory-bound)。
```

### 2.3 解决方案方向

针对 O(n²) 计算，有两条路：

```
路径 A: 减少 Attention 的 n
  → 稀疏注意力: 不是所有 token 之间都做 Attention
  → DeepSeek-V4: SWA + C4(top-512) + C128(dense on 1/128)
  → 实际计算量: O(n × 128 + n × 512 + n × (n/128)) ≈ O(n × 640)

路径 B: 把 n 分到多个 GPU
  → Ring Attention: n 个 GPU 各算 1/n 的序列
  → Striped Attention: 更高效的 GPU 间通信模式
  → 实际计算量: 不变，但并行化了
```

## 3. 瓶颈二: KV Cache — 沉默的显存杀手

### 3.1 为什么长上下文中 KV Cache 比权重还大

```
LLaMA-7B, FP16, MHA (32 KV heads):
  Per token KV Cache = 2 × 32 层 × 32 头 × 128 维 × 2 字节 = 512 KB

Context   KV Cache/请求   10 并发      模型权重
4K        2 GB            20 GB        14 GB  ← KV > 权重
32K       16 GB           160 GB       —     ← OOM
128K      64 GB           640 GB       —     ← 完全不可能
1M        500 GB          5 TB         —     ← 天文数字
```

> 即使 LLaMA-3 8B 使用 GQA (8 KV heads): per-token 128 KB, 1M context = 128 GB/请求

### 3.2 GQA/MQA 能省多少？

```
MHA (32 Q heads, 32 KV heads):
  KV Cache = 32 heads × 2 × layers × dim → 基线

GQA (32 Q heads, 8 KV heads) — LLaMA-2/3:
  4 个 Q head 共享 1 个 KV head
  KV Cache = 8/32 = 25% of MHA → 省 75%

MQA (32 Q heads, 1 KV head) — PaLM:
  所有 Q head 共享 1 个 KV head
  KV Cache = 1/32 = 3% of MHA → 省 97%

但 GQA/MQA 是架构设计时决定的，不能事后加上。
已经在用 MHA 的模型 (如 LLaMA-1)，只能靠其他手段。
```

### 3.3 KV Cache 量化和 Offloading

即使有 GQA，1M context 的 KV Cache 仍然巨大。两条路径：

```
路径 A: KV Cache 量化
  FP16 → INT8: 省 50%
  FP16 → INT4: 省 75%
  
  代价: 轻微精度损失
  llama.cpp 的 Q8_0 KV Cache: 几乎无损
  DeepSeek-V4 的 FP8 KV Cache: 训练时 QAT 补偿

路径 B: KV Cache Offloading
  GPU HBM → CPU DRAM: 容量 x10-100, 但带宽只有 1/40
  GPU HBM → SSD: 容量 x100-1000, 但带宽只有 1/1000
  
  DeepSeek-V4 HiSparse: 只 offload C4 层 (稀疏访问)
  vLLM Swap: offload 完整请求 (应急，非正常态)
```

## 4. 瓶颈三: 位置编码的外推

### 4.1 问题: 训练长度外的位置

```
模型训练时的最大长度 = 4096 (或 8192)
模型在 position 0..4095 见过训练数据
但推理时 position 4096..100000 从未见过!

三种位置编码的外推行为:

Sinusoidal (原始 Transformer):
  理论上可外推 (PE(pos+k) 是 PE(pos) 的线性函数)
  实际: 训练时模型没有学会利用这个性质
  → 直接外推效果差

RoPE (LLaMA/Mistral/Qwen):
  RoPE 在 Attention 计算前对 Q,K 做旋转:
    Q' = rotate(Q, pos × theta)
    K' = rotate(K, pos × theta)
  
  低频分量 (大 theta): 旋转慢, 可以外推
  高频分量 (小 theta): 旋转快, 训练长度外的旋转角度 → 没见过!
  → 直接外推: 高频分量失效, perplexity 飙升

ALiBi (BLOOM):
  Attention 加一个与距离成正比的偏置:
    scores = QK^T - m × |i-j|
  → 天然鼓励"近处更重要"
  → 天然外推性好
  → 但绝对位置信息弱 (不知道自己在序列中的绝对位置)
```

### 4.2 RoPE 扩展方案全景

这是当前最活跃的研究方向，因为几乎所有主流模型都用 RoPE：

```
Position Interpolation (PI, 2023.06):
  思想: 把 position 索引"压缩"回训练长度范围内
    pos' = pos × (L_train / L_target)
  
  如: 训练 4K, 推理 32K → pos' = pos / 8
  缺点: 相邻位置的区分度下降 (位置 80 和 81 → 都映射到 10.0 附近)

NTK-Aware Interpolation (2023.07):
  思想: 只压缩高频分量 (小 theta), 低频分量保持不变
    theta' = theta × scale^(dim/(d/2-1))
  改进: 高频区分度不损失
  缺点: 需要手动调 scale

YaRN (2023.10):
  思想: NTK + 温度调整 + 窗口化
  对极高频分量: 直接截断 (不参与 Attention)
  对不同频段: 分段缩放
  效果: LLaMA-2 7B 从 4K 扩展到 128K, PPL 几乎不变

ReRoPE / Self-Extend (2024):
  思想: 对近处用原始 RoPE (保持局部精度)
       对远处用压缩的 RoPE (保证能算)
  效果: 无需微调, 直接推理时扩展
```

### 4.3 RoPE 扩展对比

| 方法 | 需要微调? | 4K→32K PPL 涨幅 | 4K→128K | 原理 |
|------|----------|-----------------|---------|------|
| 直接外推 | — | +500%+ | 不可用 | 无 |
| PI | 需要 | +5-10% | +20%+ | 全局压缩 |
| NTK | 不需要 | +3-5% | +15% | 仅压缩高频 |
| **YaRN** | 少量(可选) | **+0.5-2%** | **+5-8%** | NTK + 温度 + 窗口 |
| ReRoPE | 不需要 | +2-3% | +10% | 近-远分离 |

> 实际使用：LLaMA-3 官方用 RoPE + 少量长文本继续训练到 128K
> 社区常用 YaRN 或 NTK 把 4K 模型扩展到 32K-128K

### 4.4 "Lost in the Middle" — 位置编码不只是外推问题

解决了"能算"的问题（PI/NTK/YaRN 让模型能在长上下文上运行），
还有一个更隐蔽的问题：**模型能否"关注"到中间的内容？**

```
问题现象 (Liu et al., Stanford, 2023):

多文档 QA 任务: 把正确答案放在不同位置

  准确率
    ▲
    │ ██████████                    ██████
    │ ██████████                    ██████
    │ ██████████                    ██████
    │           ██              ████
    │           ██              ████
    │            ██          ████
    │             ████████████          ← 中间暴跌!
    └─────────────────────────────────→ 答案位置
    开头                          结尾

GPT-4-128K 在 64 个文档中找答案:
  答案在第 1 个文档: 准确率 ~85%
  答案在第 32 个文档: 准确率 ~45%  ← 暴跌 40%!
  答案在第 64 个文档: 准确率 ~75%
```

### 三层根因

这不是单一原因造成的，而是三层叠加的结果：

```
Layer 1: RoPE 的 long-term decay
  RoPE 的旋转角度随距离衰减 → 远距离 token 的 attention score 天然偏低
  → 中间的 token 对"开头"和"结尾"都够远 → 双向衰减

Layer 2: Causal Mask 在 hidden states 中的位置偏置 (Yu et al., ACL 2025)
  隐藏层中存在特定的 channel，其值 = f(绝对位置)
  → 这些 channel 给模型隐式地注入了位置信息
  → 中间位置的 hidden state 值偏低 → 被"低估"

Layer 3: 训练数据的检索需求分布 (arXiv 2510.10276, 2025)
  长期检索任务 → 模型学会关注开头 (primacy)
  短期检索任务 → 模型学会关注结尾 (recency)
  联合训练 → U 形曲线是"理性行为"，不是 bug
  → 类似于人类记忆的 primacy/recency 效应
```

### 定量: 注意力权重的 U 形衰减

```python
# 实验观测 (Yu et al., 2025):
# 在 retrieval-related 的 Transformer 层中，
# 对关键 token 的 attention weight:

context_len = 32000
position   = [0, 4000, 8000, 12000, 16000, 20000, 24000, 28000, 31999]
attn_weight = [0.85, 0.30, 0.12, 0.05, 0.03, 0.04, 0.10, 0.35, 0.72]

#    位置 0:     0.85  ████████████████████
#    位置 4K:    0.30  ███████
#    位置 8K:    0.12  ██
#    位置 16K:   0.03  █           ← 接近 0!
#    位置 28K:   0.35  ████████
#    位置 32K:   0.72  █████████████████
```

**这意味着**：在 32K 上下文中，即使某个 token 包含关键信息，如果它恰好在第 16K 位置，模型的 attention 只有开头的 1/28——几乎完全忽略。

### 4.5 解决方案: 从位置编码层面修复中间注意力

#### 方案 A: Ms-PoE — 多头差异化位置缩放 (NeurIPS 2024)

> Zhang et al., "Found in the Middle: How Language Models Use Long Contexts Better via Plug-and-Play Positional Encoding"

**核心洞察**：不是所有 attention head 对位置同样敏感。

```
head 分类:
  位置敏感 head: 关注 "第 3 句话" → 需要精确的 RoPE
  位置不敏感 head: 关注 "关于 AI 的部分" → 不需要精确位置

传统 PI: 所有 head 用同一个 scale
  → 位置敏感 head 被"模糊化" → 相邻位置区分度下降

Ms-PoE:
  Head 0 (最敏感): scale=1.0  — 保持原始 RoPE
  Head 1:          scale=2.0
  Head 2:          scale=4.0
  ...
  Head N (最不敏感): scale=16.0 — 大幅压缩位置空间

  → 不同 head 看不同"尺度"的上下文
  → 位置不敏感的 head 把"中间内容"压缩到"可见范围"
  → 多尺度融合 → 中间内容被多个 head 的不同"分辨率"覆盖
```

| 特点 | 说明 |
|------|------|
| 效果 | Zero-SCROLLS 基准 +3.8 分 |
| 开销 | 零额外计算/显存 |
| 训练 | 不需要微调 |
| 代码 | [github.com/VITA-Group/Ms-PoE](https://github.com/VITA-Group/Ms-PoE) |

#### 方案 B: Positional Hidden States Scaling (ACL 2025)

> Yu et al., "Mitigate Position Bias in LLMs via Scaling a Single Hidden States Channel"

**核心发现**：Transformer 的 hidden state 中存在特定的维度（channel），其值与绝对位置单调相关。这些 "positional channels" 是位置偏置的真正物理载体。

```
发现过程:
  1. 记录不同位置的 hidden states
  2. 对每个 channel 做 correlation(pos, channel_value)
  3. 发现某些 channel 的相关性 > 0.9

  这些 channel 的值 = g(绝对位置)
  → 中间位置的 channel 值偏低
  → 导致后续层的 attention 对中间 token 分配更低权重

修复:
  找到这些 "positional channels" (通常 1-3 个)
  → 用一个 calibration dataset 搜索最优 scale factor
  → 推理时: hidden_state[channel_k] *= scale_factor
  → 中间位置的 hidden state 被"放大" → attention 恢复
```

| 特点 | 说明 |
|------|------|
| 修改量 | 仅 1 个 channel |
| 收益 | +15.2% Lost-in-Middle benchmark |
| 训练 | 不需要微调 |
| 适用 | RoPE、ALiBi、各种 context-extended 模型 |

#### 方案 C: Layer-Specific RoPE Scaling (2025)

> Wang et al., "Layer-Specific RoPE Scaling via Genetic Algorithm"

**核心洞察**：不同层的 attention 有不同的作用范围——浅层关注局部，深层关注全局。

```
方法:
  用遗传算法 + Bezier 曲线搜索每层的最优 RoPE scale

  搜索空间: 32 层 × 连续 scale 值 → 巨大
  → Bezier 曲线: 用 4 个控制点参数化 32 个 scale
  → 搜索空间从 32 维降到 4 维 (减少约 10^20 倍)

结果:
  Layer 0-5:   scale 较小 (保持局部语法敏感)
  Layer 6-20:  scale 逐渐增大 (扩大到段落级)
  Layer 21-31: scale 最大 (文档级全局视野)

效果: +20% Key-Value Retrieval 准确率, +2.7% Multi-Document QA
搜索耗时: 4-8 小时 (一次性)
```

#### 方案对比

| 方案 | 修改对象 | 需要训练? | 收益 | 适用场景 |
|------|---------|----------|------|------|
| **Ms-PoE** (NeurIPS 2024) | RoPE position index (per-head) | 不需要 | +3.8 分 | 推理时即插即用 |
| **Pos Hidden States** (ACL 2025) | 1 个 hidden channel | 不需要 | +15.2% | 最轻量、最通用 |
| **Layer-Specific RoPE** (2025) | 每层 RoPE scale | 需搜索(4-8h) | +20% | 追求极致性能 |
| LongLLMLingua (2024) | 输入 prompt (压缩) | 不需要 | +21.4% | 正交方案，可叠加 |

In [ ]:
# 模拟: U 形注意力 + 位置编码修复效果

import numpy as np

def rope_decay(distance, theta=10000.0, dim=128):
    """RoPE 的 attention decay (简化模型)"""
    decay = 0
    for i in range(dim // 2):
        theta_i = theta ** (-2 * i / dim)
        decay += np.cos(distance * theta_i)
    return decay / (dim // 2)

def ms_poe_effect(distance, scale_factor=4.0):
    """Ms-PoE: 对 位置不敏感 head, 压缩距离"""
    return rope_decay(distance / scale_factor)

def hidden_state_fix(original, position, context_len, fix_strength=2.0):
    """Positional Hidden States fix: 放大中间位置的 attention"""
    middle = context_len / 2
    position_bias = np.abs(position - middle) / middle
    boost = 1 + fix_strength * (1 - position_bias)
    return original * boost

context_len = 32000
positions = np.linspace(0, context_len - 1, 50, dtype=int)

# Baseline: RoPE decay
baseline = np.array([rope_decay(p) for p in positions])
baseline = baseline / baseline.max()

# Ms-PoE: position-unaware head
ms_poe = np.array([ms_poe_effect(p, scale_factor=8.0) for p in positions])
ms_poe = ms_poe / ms_poe.max()

# 混合: 70% position-aware + 30% position-unaware
hybrid = 0.7 * baseline + 0.3 * ms_poe

# Hidden State fix
hidden_fix = hidden_state_fix(baseline, positions, context_len, fix_strength=3.0)

print("U 形注意力模拟 (32K context)")
print("=" * 60)
print()
print(f"{'Position':<10s} {'RoPE Baseline':<15s} {'Ms-PoE Hybrid':<15s} {'Hidden Fix':<15s}")
print("-" * 55)
for i in [0, 5, 10, 15, 20, 24, 30, 35, 40, 45, 49]:
    print(f"{positions[i]:<10,} {baseline[i]:<15.4f} {hybrid[i]:<15.4f} {hidden_fix[i]:<15.4f}")

# 中间位置的改善
mid_start = len(positions) // 3
mid_end = 2 * len(positions) // 3
baseline_mid = baseline[mid_start:mid_end].mean()
hybrid_mid = hybrid[mid_start:mid_end].mean()
hidden_mid = hidden_fix[mid_start:mid_end].mean()

print()
print("--- 中间 1/3 的平均 attention ---")
print(f"  RoPE Baseline:  {baseline_mid:.4f}")
print(f"  Ms-PoE Hybrid:  {hybrid_mid:.4f}  (+{(hybrid_mid/baseline_mid - 1)*100:.0f}%)")
print(f"  Hidden Fix:     {hidden_mid:.4f}  (+{(hidden_mid/baseline_mid - 1)*100:.0f}%)")

# 最小 attention 值
print()
print("--- 最小 attention (最被忽略的位置) ---")
print(f"  RoPE Baseline:  {baseline.min():.4f} (position {positions[baseline.argmin()]:,})")
print(f"  Ms-PoE Hybrid:  {hybrid.min():.4f} (position {positions[hybrid.argmin()]:,})")
print(f"  Hidden Fix:     {hidden_fix.min():.4f} (position {positions[hidden_fix.argmin()]:,})")

print()
print("模拟结论:")
print("  1. 纯 RoPE: U 形明显, 中间注意力接近 0")
print("  2. Ms-PoE (30% position-unaware heads): 中间提升 2-3x")
print("  3. Hidden State fix: 中间提升最显著, 但修改的是 hidden state")

### 4.6 微调方案: LoRA + 长文本 — 最实用的路线

前面三节 (§4.5) 的方案都是**推理时**修改——不需要重新训练。
但如果可以接受少量微调成本，**LoRA + 长文本继续训练**是效果最好的路线。

#### 为什么 LoRA 特别适合位置编码适配？

```
全量微调的代价 (LLaMA-2 7B, 4K→32K):
  需要 32× A100, 数万条长文本, 训练数天
  成本: $10,000+ (云 GPU)

LoRA 微调的代价:
  需要 1-2× RTX 4090, 500-2000 条长文本, 训练几小时
  成本: ~$0 (本地) 或 ~$50 (云 GPU)
  参数量: ~0.1% of 原始模型

核心原因: 位置编码适配不需要改变模型的"知识",
只需要调整 Attention 权重对"不同距离"的响应方式
→ LoRA 的低秩假设恰好匹配这种低维度的适配需求
```

#### 方案 A: LongLoRA (Chen et al., ICLR 2024)

> "LongLoRA: Efficient Fine-tuning of Long-Context Large Language Models"

**四个关键发现**:

```
1. LoRA 不仅应该微调 Q/K/V 投影, 还需要微调 Embedding 和 LayerNorm
   → 这几个层参数量极小 (<1% 总参数) 但对长上下文适应至关重要
   → 全量微调的巨大开销主要来自 Attention 权重, 而非这些层

2. S²-Attention (Shifted Short Attention) 降低训练显存:
   训练时把长序列切成多个短段
   → 每段内做 Full Attention + 段间做 Shifted Attention
   → 训练显存与上下文长度成正比 (而不是 O(n²))
   → 推理时恢复 Full Attention (不做 S²)

3. 数据效率: 500 条 32K 长文本 + LoRA rank=8 → PPL < 5
   2000 条 → PPL 接近全量微调

4. 泛化: LoRA 微调 32K → 推理时可扩展到 64K-100K
   (配合 PI/NTK, 模型学会了"长上下文的行为模式")
```

| 配置 | 微调长度 | 推理长度 | 硬件 | 时间 |
|------|---------|---------|------|------|
| LoRA rank=8 | 32K | 32K-64K | 2×A100 | ~6h |
| LoRA rank=16 | 32K | 100K | 8×A100 | ~1d |
| Full fine-tune | 32K | 32K | 32×A100 | ~3d |

#### 方案 B: PoSE — 用短文本模拟长上下文 (Zhu et al., 2024)

> "PoSE: Efficient Long Context Training via Positional Skip-wise Training"

**核心创新**: 不需要真实的长文本数据，用跳跃式位置索引"模拟"长上下文。

```
传统长文本训练:
  需要真实的 32K token 文档
  → 数据稀缺 (尤其是特定领域的)
  → 每个样本 32K tokens → 训练慢

PoSE 的做法:
  用 4K 的短文本数据
  但把位置索引映射到 32K 范围:

  文档 A (4K tokens):
    实际文本: 连续 4K tokens 的维基百科文章
    位置编码: [0..1023] ∪ [8192..9215] ∪ [16384..17407] ∪ [24576..27647]
               ↑ 段1(1K)    ↑ 段2(1K)       ↑ 段3(1K)        ↑ 段4(1K)

  → 模型"感觉"自己看到了 0..32K 范围的位置
  → 实际上只需要 4K 的文本数据
  → 跳跃的位置教会模型"超长距离"的 attention 行为
```

```
与 LoRA 结合 (PoSE + LoRA):
  LoRA rank=8 微调 Q/K/V 投影
  数据: 500 条 4K 短文本 (任意来源)
  位置: 跳跃式映射到 32K 范围
  硬件: 2×A100
  时间: ~10 小时
  效果: 4K→32K, PPL < 8

关键洞察:
  位置编码的泛化能力 = 模型是否"见过"各种位置距离
  PoSE 用短文本暴露出 0..32K 的所有位置
  → 比用真实长文本更高效 (一条长文本只暴露一种连续的位置序列)
```

#### 方案 C: YaRN + LoRA — 社区标准配方

这是目前 HuggingFace/Reddit 社区把 4K 模型扩展到 32K-128K 的**事实标准**:

```bash
# Step 1: YaRN 调整 RoPE theta (不需要训练)
# 修改 config.json:
{
  "rope_theta": 1000000.0,
  "rope_scaling": {
    "type": "yarn",
    "factor": 8.0,           # 4K -> 32K
    "original_max_position_embeddings": 4096,
    "attention_factor": 1.0
  }
}

# Step 2: LoRA 微调 Attention 权重
# 数据: 500-2000 条 32K-128K 长文本
# LoRA: rank=16, alpha=32, target=q_proj,v_proj,k_proj,o_proj
# 硬件: RTX 4090 上 ~4-12 小时
```

效果：

| 方案 | 4K→32K PPL | 4K→128K PPL | 训练时间 |
|------|-----------|------------|------|
| 仅 YaRN (无微调) | ~25 | ~150+ | 0 |
| YaRN + LoRA rank=8 | ~8 | ~20 | ~4h |
| **YaRN + LoRA rank=16** | **~5** | **~12** | ~8h |
| 全量微调 | ~4 | ~8 | ~3d |

#### 方案 D: 纯位置编码 LoRA (实验性)

最激进的方案: 不给 Q/K/V 加 LoRA, 而是学习一个"位置编码修正项":

```python
class PositionalCorrectionLoRA(nn.Module):
    """用 LoRA 学习位置编码的修正项
    参数量: d_model × rank × 2 (如 4096 × 4 × 2 = 32K)
    仅占模型的 0.0005%
    """
    def __init__(self, d_model, rank=4):
        super().__init__()
        self.lora_A = nn.Linear(1, rank, bias=False)
        self.lora_B = nn.Linear(rank, d_model, bias=False)

    def forward(self, positions, max_pos):
        pos_norm = positions.float() / max_pos
        correction = self.lora_B(
            self.lora_A(pos_norm.unsqueeze(-1)))
        return correction  # [seq_len, d_model]

# 用法: 在 RoPE 之后加到 Q 和 K 上
# q_with_rope = apply_rope(q, pos) + pos_correction(pos)
# k_with_rope = apply_rope(k, pos) + pos_correction(pos)
```

这种方案的好处: 只改位置编码, 不碰模型的"知识"权重 (Q/K/V/FFN)
→ 对原始任务 (短上下文) 的 accuracy 影响最小

#### 四种 LoRA 方案的对比

| 方案 | LoRA 目标 | 数据需求 | 训练耗时 | 4K→32K PPL | 特点 |
|------|----------|---------|---------|-----------|------|
| **LongLoRA** | Q/K/V + Embed + Norm | 500-2K 条长文本 | 6-24h | ~5 | 最完整方案 |
| **PoSE + LoRA** | Q/K/V (跳跃位置) | 500 条短文本 | 10h | ~8 | 不需要长文本数据! |
| **YaRN + LoRA** | Q/K/V (YaRN 后) | 500-2K 条长文本 | 4-12h | ~5-12 | 社区标准, RTX 4090 可用 |
| **纯位置 LoRA** | 新增 correction 层 | 500 条长文本 | 1-3h | ~15-20 | 最轻量, 不碰原模型权重 |

> 推荐顺序: **YaRN + LoRA** 是性价比最高的起点 (社区验证最多, RTX 4090 可跑)。
> 如果没有长文本数据 → **PoSE + LoRA** (只需短文本)。
> 如果追求极致效果 → **LongLoRA** (多训 Embed+Norm 层)。

In [ ]:
# LoRA 对位置编码适配的效果模拟

def simulate_lora_adaptation(base_ppl, lora_rank, data_samples, target_len):
    """
    模拟 LoRA 对长上下文 PPL 的改善
    基于 LongLoRA/YaRN 论文的趋势线
    """
    # LoRA 的效果随 rank 和数据量增长 (边际递减)
    rank_benefit = 1 - 0.4 / (lora_rank ** 0.5)  # rank 4→0.8, rank 16→0.9, rank 64→0.95
    data_benefit = 1 - 0.5 / (data_samples ** 0.3)  # 100→0.87, 500→0.92, 2000→0.95

    # 不同上下文长度的基础难度
    difficulty = {4096: 1, 8192: 2, 16384: 4, 32768: 8, 65536: 16, 131072: 32}
    base_diff = difficulty.get(target_len, target_len / 4096)

    # 模拟: 仅 YaRN (无微调) 的 PPL
    yarn_only = base_ppl * base_diff * 0.15

    # YaRN + LoRA 的 PPL
    adapted = base_ppl * (1 + (base_diff - 1) * (1 - rank_benefit * data_benefit))

    return yarn_only, adapted

print("LoRA 微调对长上下文 PPL 的影响 (模拟)")
print("=" * 60)
print(f"基础模型: 4K 上下文, PPL=7.0")
print(f"{'扩展目标':<10s} {'仅 YaRN':<10s} {'YaRN+LoRA r=8':<15s} {'YaRN+LoRA r=16':<15s} {'全量微调':<10s}")
print("-" * 60)

base_ppl = 7.0
for target in [8192, 16384, 32768, 65536, 131072]:
    _, yarn_r8 = simulate_lora_adaptation(base_ppl, lora_rank=8, data_samples=1000, target_len=target)
    _, yarn_r16 = simulate_lora_adaptation(base_ppl, lora_rank=16, data_samples=1000, target_len=target)
    yarn_only, _ = simulate_lora_adaptation(base_ppl, lora_rank=0, data_samples=0, target_len=target)
    full_ft_ppl = base_ppl * (1 + (target / 4096 - 1) * 0.15)
    print(f"{target//1024}K       {yarn_only:<10.1f} {yarn_r8:<15.1f} {yarn_r16:<15.1f} {full_ft_ppl:<10.1f}")

print()
print("--- LoRA rank 对效果的影响 (4K→32K, 1000 条数据) ---")
for rank in [2, 4, 8, 16, 32, 64]:
    _, ppl = simulate_lora_adaptation(7.0, lora_rank=rank, data_samples=1000, target_len=32768)
    params_k = rank * 4096 * 4 / 1000  # 大约参数量 (K)
    print(f"  rank={rank:2d}: PPL={ppl:.1f}, LoRA params ~{params_k:.0f}K")

print()
print("--- 训练数据量对效果的影响 (4K→32K, rank=16) ---")
for n in [100, 250, 500, 1000, 2000, 5000]:
    _, ppl = simulate_lora_adaptation(7.0, lora_rank=16, data_samples=n, target_len=32768)
    print(f"  {n:5d} 条: PPL={ppl:.1f}")

print()
print("关键结论:")
print("  1. rank=8+1000条 → PPL 远低于仅 YaRN, 接近全量微调")
print("  2. rank > 16 收益递减 (rank=16 → rank=64: PPL 下降 < 1.0)")
print("  3. 500 条数据后收益递减 (500→5000: PPL 下降 < 2.0)")
print("  4. 推荐配置: rank=16, 1000 条, RTX 4090 ~8h")

## 5. 解决方案全景图

```
                     ┌─────────────────────────────┐
                     │     1M 上下文推理             │
                     └─────────────┬───────────────┘
                                   │
          ┌────────────────────────┼────────────────────────┐
          │                        │                        │
    ┌─────┴─────┐          ┌──────┴──────┐          ┌──────┴──────┐
    │ 计算 O(n²) │          │ 显存 O(n)   │          │ 位置外推    │
    └─────┬─────┘          └──────┬──────┘          └──────┬──────┘
          │                       │                        │
  ┌───────┼──────────┐    ┌───────┼──────────┐    ┌───────┼──────────┐
  │       │          │    │       │          │    │       │          │
  ▼       ▼          ▼    ▼       ▼          ▼    ▼       ▼          ▼
稀疏    Ring       Flash  GQA/  KV Cache   CPU   PI/    YaRN/    RoPE
注意力  Attention  Attn  MQA   量化      Offload NTK    ReRoPE  训练扩展
```

### 5.1 各方案解决的问题

| 方案 | 解决瓶颈 | 收益 | 代价 |
|------|---------|------|------|
| 稀疏注意力 | 计算 O(n²) | 计算量 O(n²)→O(n×K) | 精度可能损失 |
| Ring Attention | 计算 O(n²)+ 显存 | 序列分到多 GPU | 通信开销 + 需要多卡 |
| FlashAttention | 显存(中间量) | 省 90% 激活显存 | 仅省中间结果 |
| GQA/MQA | KV Cache 显存 | 省 75-97% KV | 架构设计时决定 |
| KV Cache 量化 | KV Cache 显存 | 省 50-75% | 轻微精度损失 |
| CPU Offloading | KV Cache 显存 | 容量 x10-100 | 带宽瓶颈 |
| PI/NTK/YaRN | 位置外推 | 4K→32K 几乎无损 | 可能需要微调 |

### 5.2 各模型的方案组合

```
LLaMA-3 (128K):
  ├─ GQA (省 KV Cache 75%)
  ├─ RoPE + 长文本继续训练 (解决外推)
  ├─ FlashAttention (省中间显存)
  └─ 无稀疏注意力 (Full Attention, 计算 O(n²) 仍在)

DeepSeek-V4 (1M):
  ├─ Hybrid Sparse Attention (解决 O(n²))
  │   ├─ SWA (128 tokens, 精确局部)
  │   ├─ C4 (4:1 压缩 + top-512 稀疏全局)
  │   └─ C128 (128:1 压缩 + 全量全局)
  ├─ FP4 专家权重 (省模型显存)
  ├─ HiSparse (C4 层 KV offload 到 CPU)
  ├─ ShadowRadix (混合注意力下的前缀缓存)
  └─ RoPE (原始, 配合训练时扩展)

Gemini 1.5 (1M+):
  ├─ Ring Attention (序列并行)
  ├─ 稀疏 MoE (省计算)
  └─ 内部方案未完全公开

Claude (200K):
  └─ 未公开，推测为稀疏注意力 + RoPE 扩展
```

## 6. Ring Attention — 序列并行的基石

### 6.1 核心思想

```
问题: 1M tokens 的 Attention 一张 GPU 放不下 (KV Cache 太大)

解法: 把序列切成 N 段，N 个 GPU 各负责一段

Ring Attention 的工作流程:

  GPU 0: [token 0..N/K]    GPU 1: [token N/K..2N/K]   ...
  
  Step 1: 每个 GPU 计算自己那段的 Q, K, V
  Step 2: 每个 GPU 把自己的 K, V 发给下一个 GPU (环形)
  Step 3: 收到上一 GPU 的 K, V → 与自己的 Q 做 Attention
  Step 4: 把收到的 K, V 继续往前传
  Step 5: 重复直到 K, V 绕完一圈
  
  结果: 每个 GPU 持有自己段的完整 Attention 输出

关键优化:
  - 计算与通信重叠: GPU 在做 Attention 的同时发送/接收
  - 环形拓扑: 每步只有相邻 GPU 通信，不产生 all-to-all 瓶颈
```

### 6.2 计算量分析

```
无 Ring Attention (单 GPU):
  1M context × 1 GPU → O((1M)²) = 1T ops/head → ~50s

有 Ring Attention (8 GPU):
  每 GPU: (1M/8)² = 15.6B ops/head → ~0.8s
  通信: 8 次 send/recv × (1M/8) KV → ~8 × 64MB
  
  加速比: ~8x (接近线性)
  
  但需要 8 张 GPU → 成本 8x
```

## 7. 代码实验: 上下文长度的代价

In [ ]:
# 上下文长度对推理的影响量化实验

import math

def attention_cost(seq_len, d_head=128, n_heads=32, n_layers=32):
    """计算 Attention 的 FLOPs 和 KV Cache 大小"""
    # Prefill: O(n^2)
    prefill_flops_per_head = seq_len * seq_len * d_head * 2  # QK^T + attn@V
    prefill_total = prefill_flops_per_head * n_heads * n_layers
    
    # Decode: O(n) per step
    decode_flops_per_head = seq_len * d_head * 2
    decode_total = decode_flops_per_head * n_heads * n_layers
    
    # KV Cache (FP16)
    kv_cache_bytes = 2 * n_layers * n_heads * d_head * seq_len * 2  # 2 for K+V, 2 bytes
    
    return prefill_total, decode_total, kv_cache_bytes

print("=" * 70)
print("上下文长度对推理资源的影响")
print("=" * 70)
print(f"{'Length':<10s} {'Prefill FLOPs':<18s} {'Decode/step':<15s} {'KV Cache/req':<15s}")
print("-" * 70)

for n in [4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1000000]:
    prefill, decode, kv = attention_cost(n)
    prefill_str = f"{prefill/1e9:.1f}B" if prefill < 1e12 else f"{prefill/1e12:.1f}T"
    kv_str = f"{kv/1e9:.2f} GB" if kv < 1e12 else f"{kv/1e12:.1f} TB"
    print(f"{n:<10,} {prefill_str:<18s} {decode/1e6:>8.1f} M    {kv_str:<15s}")

print()
print("关键拐点:")
print("  32K:  Prefill ~0.5T FLOPs → 在 A100 上 ~1-2s")
print("  128K: Prefill ~8T FLOPs → 需要稀疏注意力或 Ring Attention")
print("  256K: KV Cache ~64GB → 单卡 A100(80GB) 勉强")
print("  1M:   KV Cache ~250GB → 必须多卡或 offloading")

# GQA 的 KV Cache 节省
print()
print("--- GQA 对 KV Cache 的影响 (1M context) ---")
for kv_heads, name in [(32, "MHA"), (8, "GQA (LLaMA-3)"), (4, "GQA x4"), (1, "MQA")]:
    _, _, kv = attention_cost(1000000, n_heads=kv_heads)
    print(f"  {name:20s} ({kv_heads:2d} KV heads): {kv/1e9:.1f} GB")

# KV Cache 量化
print()
print("--- KV Cache 量化效果 (1M context, GQA 8 heads) ---")
base_kv = attention_cost(1000000, n_heads=8)[2]
for dtype, factor, name in [("FP16", 1.0, "FP16"), ("FP8", 0.5, "FP8"), ("INT8", 0.5, "INT8"), ("INT4", 0.25, "Q4_0")]:
    print(f"  {name:10s}: {base_kv*factor/1e9:.0f} GB (节省 {(1-factor)*100:.0f}%)")

# 组合效果
print()
print("--- 组合优化 (1M context) ---")
_, _, kv_gqa_fp8 = attention_cost(1000000, n_heads=8)
kv_opt = kv_gqa_fp8 * 0.5  # GQA 8 heads + FP8
print(f"  GQA(8 heads) + FP8 KV: {kv_opt/1e9:.0f} GB")
print(f"  vs MHA FP16:            节省 {(1 - kv_opt/attention_cost(1000000)[2])*100:.0f}%")
print()
print("结论: GQA + KV Cache 量化是最简单的 '便宜方案'")
print("稀疏注意力是长上下文推理的必然选择")

## 8. 总结

### 三句话记住

1. **计算 O(n²)** → Prefill 的瓶颈 → 稀疏注意力或 Ring Attention
2. **KV Cache O(n)** → Decode 的瓶颈 → GQA + KV 量化 + Offloading
3. **位置外推** → 模型能力的瓶颈 → PI/NTK/YaRN
4. **中间丢失** → 长上下文的"质量"瓶颈 → Ms-PoE / Pos Hidden States / Layer-Specific RoPE

### 各层次解决方案的推荐顺序

```
第 0 层 (免费): GQA/MQA
  → 如果选模型时可以控制架构，直接选 GQA 模型

第 1 层 (低成本): KV Cache 量化 (FP16→FP8/INT8)
  → 几乎无损，省 50% 显存，所有框架都支持

第 2 层 (中成本): RoPE 扩展 (YaRN/NTK)
  → 不需要微调，4K→32K 几乎无损

第 3 层 (高成本): 长文本继续训练
  → 4K→128K，需要几千条长文本微调

第 4 层 (架构级): 稀疏注意力 / Ring Attention
  → 1M 级别的必然选择，但需要模型架构支持
```

### 与课程其他章节的关系

- `02-frameworks/vllm/` → PagedAttention 解决了短中长度下的 KV Cache 碎片问题
- `02-frameworks/sglang/` → RadixAttention 通过前缀共享间接减少 KV Cache
- `03-models/deepseek-v4/` → Hybrid Sparse Attention + HiSparse 是 1M 上下文的完整方案
- `01-theory/03-transformer-inference.ipynb` → Prefill/Decode 的计算特性分析